# Investigating Surprising Results from Dimreduce Comparison

Four surprises to explain:
1. AE preserves distances/neighborhoods better than t-SNE/UMAP
2. t-SNE+MLP gives best reconstruction at n=16
3. PCA+MLP vs AE+MLP reconstruction at n=5
4. Deep AE (l=3) worse than l=1 at n=5

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
import umap
import warnings
warnings.filterwarnings('ignore')

from core import (
    Autoencoder, generate_sparse_data, train_autoencoder,
    run_experiment_multi_seed, device
)
print(f"Device: {device}")

In [ ]:
# Reproduce the models and data from the main notebook
S = 0.95
n_viz = 2000

configs = [
    {'n': 5, 'm': 2, 'l': 1, 'label': 'n=5, l=1'},
    {'n': 5, 'm': 2, 'l': 3, 'label': 'n=5, l=3'},
    {'n': 16, 'm': 2, 'l': 1, 'label': 'n=16, l=1'},
    {'n': 16, 'm': 2, 'l': 3, 'label': 'n=16, l=3'},
]

results = {}
for cfg in configs:
    key = cfg['label']
    res = run_experiment_multi_seed(
        n=cfg['n'], m=cfg['m'], l=cfg['l'],
        S=S, n_seeds=10, n_steps=10000,
        importance_decay=0.7, verbose=False
    )
    results[key] = res
    print(f"{key}: loss={res['final_loss']:.6f}, nonlinear_gain={res['nonlinear_gain']:.4f}")

datasets = {}
for n in [5, 16]:
    torch.manual_seed(42)
    datasets[n] = generate_sparse_data(n_viz, n, S).cpu().numpy()

## Surprise 1: AE preserves distances/neighborhoods better than t-SNE/UMAP

**Hypothesis**: Sparse independent features don't have manifold structure. t-SNE/UMAP assume a manifold and warp space to emphasize it — when there's no manifold, they *create* distortion. The AE's linear (l=1) or near-linear encoding is essentially a projection, which preserves distances about as well as PCA.

Let's test this by:
1. Looking at what t-SNE/UMAP actually do to this data visually
2. Comparing with data that DOES have manifold structure (correlated features)
3. Checking if this is specific to high sparsity

In [ ]:
# 1a: Visualize the pairwise distance distributions to understand what's happening
def analyze_distance_distortion(x_np, label, n_sub=500):
    """Detailed look at how each method distorts distances."""
    x = x_np[:n_sub]
    d_orig = pdist(x)
    
    # Compute embeddings
    pca = PCA(n_components=2).fit_transform(x)
    tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(x)
    um = umap.UMAP(n_components=2, random_state=42).fit_transform(x)
    
    methods = {'PCA': pca, 't-SNE': tsne, 'UMAP': um}
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, z) in zip(axes, methods.items()):
        d_emb = pdist(z)
        # Normalize
        d_o = d_orig / d_orig.max()
        d_e = d_emb / d_emb.max()
        
        rho, _ = spearmanr(d_o, d_e)
        
        ax.scatter(d_o, d_e, s=1, alpha=0.1, rasterized=True)
        ax.plot([0, 1], [0, 1], 'r--')
        ax.set_title(f'{name} (r={rho:.3f})')
        ax.set_xlabel('Original dist')
        ax.set_ylabel('Embedded dist')
        ax.set_aspect('equal')
    
    fig.suptitle(f'Distance distortion: {label}', fontsize=14)
    plt.tight_layout()
    plt.show()

analyze_distance_distortion(datasets[5], 'n=5, sparse independent')
analyze_distance_distortion(datasets[16], 'n=16, sparse independent')

In [ ]:
# 1b: What does the input data actually look like?
# With S=0.95 and independent features, most samples have 0 or 1 active features.
# This is not a manifold — it's a sparse union of rays from the origin.

for n in [5, 16]:
    x = datasets[n]
    n_active = (x > 0).sum(axis=1)
    print(f"\nn={n}: distribution of # active features:")
    for k in range(min(n+1, 8)):
        frac = (n_active == k).mean()
        print(f"  {k} active: {frac:.3f} ({(n_active == k).sum()} samples)")
    print(f"  mean: {n_active.mean():.2f}")
    
    # What fraction of the data is essentially on a coordinate axis (single feature active)?
    on_axis = (n_active <= 1).mean()
    print(f"  On or near origin (0-1 active): {on_axis:.3f}")

In [ ]:
# 1c: Compare with data that HAS manifold structure.
# Use correlated features: fewer true dimensions than observed dimensions.
from core import generate_correlated_features

def neighborhood_preservation(x_np, z, k=10, n_sub=500):
    idx = np.random.RandomState(42).choice(len(x_np), min(n_sub, len(x_np)), replace=False)
    x_sub, z_sub = x_np[idx], z[idx]
    nn_x = NearestNeighbors(n_neighbors=k+1).fit(x_sub)
    nn_z = NearestNeighbors(n_neighbors=k+1).fit(z_sub)
    _, idx_x = nn_x.kneighbors(x_sub)
    _, idx_z = nn_z.kneighbors(z_sub)
    preserved = sum(len(set(idx_x[i, 1:]) & set(idx_z[i, 1:])) / k for i in range(len(x_sub)))
    return preserved / len(x_sub)

# Generate manifold-like data: 16 observed features from 3 true sources
torch.manual_seed(42)
x_manifold = generate_correlated_features(2000, 16, n_true_features=3, S=0.5).cpu().numpy()
# And compare with our sparse independent data
x_sparse = datasets[16]

print("Neighborhood preservation (k=10):")
print(f"{'Data':<25} {'PCA':<10} {'t-SNE':<10} {'UMAP':<10}")
print('-' * 55)

for x, label in [(x_sparse, 'Sparse independent'), (x_manifold, 'Correlated (3 sources)')]:
    pca_z = PCA(n_components=2).fit_transform(x)
    tsne_z = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(x)
    umap_z = umap.UMAP(n_components=2, random_state=42).fit_transform(x)
    
    p_pca = neighborhood_preservation(x, pca_z)
    p_tsne = neighborhood_preservation(x, tsne_z)
    p_umap = neighborhood_preservation(x, umap_z)
    
    print(f"{label:<25} {p_pca:<10.3f} {p_tsne:<10.3f} {p_umap:<10.3f}")

print("\nSpearman distance correlation:")
print(f"{'Data':<25} {'PCA':<10} {'t-SNE':<10} {'UMAP':<10}")
print('-' * 55)

for x, label in [(x_sparse, 'Sparse independent'), (x_manifold, 'Correlated (3 sources)')]:
    x_sub = x[:300]
    d_orig = pdist(x_sub)
    pca_z = PCA(n_components=2).fit_transform(x_sub)
    tsne_z = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(x_sub)
    umap_z = umap.UMAP(n_components=2, random_state=42).fit_transform(x_sub)
    
    for name, z in [('PCA', pca_z), ('t-SNE', tsne_z), ('UMAP', umap_z)]:
        rho, _ = spearmanr(d_orig, pdist(z))
    
    rhos = []
    for z in [pca_z, tsne_z, umap_z]:
        rho, _ = spearmanr(d_orig, pdist(z))
        rhos.append(rho)
    print(f"{label:<25} {rhos[0]:<10.3f} {rhos[1]:<10.3f} {rhos[2]:<10.3f}")

In [ ]:
# 1d: Sparsity sweep — does the AE advantage disappear at lower sparsity?
print("Neighborhood preservation (k=10) vs sparsity:")
print(f"{'S':<8} {'n':<5} {'AE (l=1)':<12} {'PCA':<10} {'t-SNE':<10} {'UMAP':<10}")
print('-' * 55)

for S_test in [0.5, 0.75, 0.95]:
    for n in [5, 16]:
        torch.manual_seed(42)
        x = generate_sparse_data(2000, n, S_test).cpu().numpy()
        
        # Train a quick AE
        res = run_experiment_multi_seed(n=n, m=2, l=1, S=S_test, n_seeds=5,
                                        n_steps=5000, importance_decay=0.7, verbose=False)
        model = res['model']
        model.eval()
        with torch.no_grad():
            z_ae = model.encode(torch.tensor(x, dtype=torch.float32, device=device)).cpu().numpy()
        
        pca_z = PCA(n_components=2).fit_transform(x)
        tsne_z = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(x)
        umap_z = umap.UMAP(n_components=2, random_state=42).fit_transform(x)
        
        p_ae = neighborhood_preservation(x, z_ae)
        p_pca = neighborhood_preservation(x, pca_z)
        p_tsne = neighborhood_preservation(x, tsne_z)
        p_umap = neighborhood_preservation(x, umap_z)
        
        print(f"{S_test:<8} {n:<5} {p_ae:<12.3f} {p_pca:<10.3f} {p_tsne:<10.3f} {p_umap:<10.3f}")

## Surprise 2: t-SNE+MLP gives best reconstruction at n=16

t-SNE embeddings + small MLP decoder gave MSE=0.0078, beating AE native (0.012) and PCA+MLP (0.011).

Possible explanations:
- t-SNE spreads points more uniformly, making it easier for an MLP to partition the space
- The MLP decoder is more powerful than the AE's decoder and the comparison is unfair
- Overfitting: the MLP is memorizing the train set

Let's check.

In [ ]:
# 2a: Is the MLP decoder overfitting?
# Check train vs test MSE for each method.

n = 16
x_np = datasets[n]
n_train = 1600
x_train, x_test = x_np[:n_train], x_np[n_train:]

# Compute embeddings
pca = PCA(n_components=2)
z_pca_all = pca.fit_transform(x_np)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
z_tsne_all = tsne.fit_transform(x_np)
um = umap.UMAP(n_components=2, random_state=42)
z_umap_all = um.fit_transform(x_np)

model = results['n=16, l=1']['model']
model.eval()
with torch.no_grad():
    z_ae_all = model.encode(torch.tensor(x_np, dtype=torch.float32, device=device)).cpu().numpy()

print(f"{'Method':<12} {'Train MSE':<14} {'Test MSE':<14} {'Ratio (test/train)'}")
print('-' * 55)

for name, z_all in [('AE', z_ae_all), ('PCA', z_pca_all), ('t-SNE', z_tsne_all), ('UMAP', z_umap_all)]:
    z_tr, z_te = z_all[:n_train], z_all[n_train:]
    
    mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42,
                       early_stopping=True, validation_fraction=0.2)
    mlp.fit(z_tr, x_train)
    
    train_mse = np.mean((x_train - mlp.predict(z_tr))**2)
    test_mse = np.mean((x_test - mlp.predict(z_te))**2)
    
    print(f"{name:<12} {train_mse:<14.6f} {test_mse:<14.6f} {test_mse/train_mse:.2f}")

In [ ]:
# 2b: CRITICAL ISSUE — t-SNE has no out-of-sample transform.
# When we split train/test and use z_tsne_all[n_train:] as "test embeddings",
# those test points were fitted jointly with training points during t-SNE.
# This means test embeddings are informed by test inputs — it's data leakage!
#
# For a fair comparison, we should:
# - For PCA/UMAP: fit on train, transform test (both support this)
# - For t-SNE: fit on train only, test points have no valid embedding
# - Or: use the SAME split for fitting t-SNE as for fitting the decoder
#
# Let's redo this properly.

print("=== FAIR COMPARISON (no data leakage) ===")
print("For t-SNE: fit on train data only, no test evaluation possible.")
print("For AE/PCA/UMAP: fit on train, transform test.\n")

n = 16
x_np = datasets[n]
n_train = 1600
x_train, x_test = x_np[:n_train], x_np[n_train:]

# PCA: fit on train, transform test
pca = PCA(n_components=2)
z_pca_tr = pca.fit_transform(x_train)
z_pca_te = pca.transform(x_test)

# UMAP: fit on train, transform test
um = umap.UMAP(n_components=2, random_state=42)
z_umap_tr = um.fit_transform(x_train)
z_umap_te = um.transform(x_test)

# AE: already has out-of-sample via encode()
for l_val, key in [(1, 'n=16, l=1'), (3, 'n=16, l=3')]:
    model = results[key]['model']
    model.eval()
    with torch.no_grad():
        z_ae_tr = model.encode(torch.tensor(x_train, dtype=torch.float32, device=device)).cpu().numpy()
        z_ae_te = model.encode(torch.tensor(x_test, dtype=torch.float32, device=device)).cpu().numpy()
        # Also get native reconstruction MSE on test
        x_recon_te = model(torch.tensor(x_test, dtype=torch.float32, device=device))[0].cpu().numpy()
        native_mse = np.mean((x_test - x_recon_te)**2)
    
    print(f"\n--- AE l={l_val} ---")
    print(f"Native decoder test MSE: {native_mse:.6f}")
    
    # Fit MLP decoders
    for name, z_tr, z_te in [('AE', z_ae_tr, z_ae_te), ('PCA', z_pca_tr, z_pca_te), ('UMAP', z_umap_tr, z_umap_te)]:
        mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42,
                           early_stopping=True, validation_fraction=0.2)
        mlp.fit(z_tr, x_train)
        train_mse = np.mean((x_train - mlp.predict(z_tr))**2)
        test_mse = np.mean((x_test - mlp.predict(z_te))**2)
        print(f"  {name}+MLP: train={train_mse:.6f}, test={test_mse:.6f}")

print("\nNote: t-SNE excluded because it has no out-of-sample transform.")
print("The original comparison was unfair — t-SNE test embeddings were computed jointly with training data.")

In [ ]:
# 2c: But wait — the original notebook DID use z_tsne_all[n_train:] as test.
# Let's quantify how much leakage this causes by comparing:
# t-SNE fit on all data (leaky) vs t-SNE fit on train only + MLP on train only

# Leaky version (what the original notebook did)
z_tsne_all = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(x_np)
z_tsne_tr_leaky = z_tsne_all[:n_train]
z_tsne_te_leaky = z_tsne_all[n_train:]

mlp_leaky = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42,
                          early_stopping=True, validation_fraction=0.2)
mlp_leaky.fit(z_tsne_tr_leaky, x_train)
leaky_train = np.mean((x_train - mlp_leaky.predict(z_tsne_tr_leaky))**2)
leaky_test = np.mean((x_test - mlp_leaky.predict(z_tsne_te_leaky))**2)

# Clean version: t-SNE on train only, use for train-set cross-val
z_tsne_tr_clean = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(x_train)

# Cross-validate on the training set itself
from sklearn.model_selection import cross_val_score
mlp_clean = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42,
                          early_stopping=True, validation_fraction=0.2)
mlp_clean.fit(z_tsne_tr_clean, x_train)
clean_train = np.mean((x_train - mlp_clean.predict(z_tsne_tr_clean))**2)

print("t-SNE reconstruction comparison:")
print(f"  Leaky (fit on all, split after):  train={leaky_train:.6f}, test={leaky_test:.6f}")
print(f"  Clean (fit on train only):         train={clean_train:.6f}, test=N/A (no transform)")
print(f"\nThe leaky test MSE ({leaky_test:.6f}) is artificially good because t-SNE")
print(f"placed test points knowing their input values.")

In [ ]:
# 2d: Even the leaky comparison deserves scrutiny. Let's also check:
# Is the MLP decoder just much more powerful than the AE's built-in decoder?
# The AE decoder for l=1 is a single linear layer + ReLU (tied weights).
# The MLP has (64, 32) hidden layers.

# Fair test: give the AE embeddings the same powerful MLP decoder
# and compare with PCA embeddings + same MLP decoder.
# This isolates the embedding quality from decoder capacity.

print("=== Controlling for decoder capacity ===")
print("Same MLP decoder (64, 32) applied to each embedding.")
print("Leaky t-SNE included for reference but marked.\n")

n = 16
# Use multiple MLP random seeds to get error bars
results_table = {}
for name, z_tr, z_te, leaky in [
    ('AE l=1', z_ae_tr, z_ae_te, False),  # uses l=1 model from above
    ('PCA', z_pca_tr, z_pca_te, False),
    ('UMAP', z_umap_tr, z_umap_te, False),
    ('t-SNE*', z_tsne_tr_leaky, z_tsne_te_leaky, True),
]:
    test_mses = []
    for seed in range(5):
        mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=seed,
                           early_stopping=True, validation_fraction=0.2)
        mlp.fit(z_tr, x_train)
        mse = np.mean((x_test - mlp.predict(z_te))**2)
        test_mses.append(mse)
    
    marker = ' (LEAKY)' if leaky else ''
    print(f"{name:<10} test MSE: {np.mean(test_mses):.6f} +/- {np.std(test_mses):.6f}{marker}")

In [ ]:
# 2e: Also check with a much larger MLP to see if we're capacity-limited
print("\n=== Larger MLP decoder (256, 128, 64) ===")
for name, z_tr, z_te, leaky in [
    ('AE l=1', z_ae_tr, z_ae_te, False),
    ('PCA', z_pca_tr, z_pca_te, False),
    ('UMAP', z_umap_tr, z_umap_te, False),
]:
    mlp = MLPRegressor(hidden_layer_sizes=(256, 128, 64), max_iter=1000, random_state=42,
                       early_stopping=True, validation_fraction=0.2)
    mlp.fit(z_tr, x_train)
    train_mse = np.mean((x_train - mlp.predict(z_tr))**2)
    test_mse = np.mean((x_test - mlp.predict(z_te))**2)
    print(f"{name:<10} train={train_mse:.6f}, test={test_mse:.6f}")

## Surprise 3: PCA+MLP reconstruction comparison

At n=5: PCA+MLP (0.000792) seems competitive with AE+MLP (0.000379). But the AE's native decoder (0.001878) is worse than PCA+MLP. What's going on?

For l=1 with tied weights, the encoder IS a linear projection (W). So AE embedding = linear projection, PCA embedding = linear projection. The question is: which linear projection is better for reconstruction?

In [ ]:
# 3a: The AE (l=1, tied weights) encoder is literally a linear map.
# Compare the actual projection matrices.

n = 5
model = results['n=5, l=1']['model']
model.eval()

# AE's encoder weight
W_ae = model.encoder.weight.detach().cpu().numpy()  # shape (m, n)
print("AE encoder weight W (m x n):")
print(W_ae)
print(f"\nSingular values of W: {np.linalg.svd(W_ae, compute_uv=False)}")

# PCA components
pca = PCA(n_components=2)
pca.fit(datasets[5])
W_pca = pca.components_  # shape (m, n)
print(f"\nPCA components:")
print(W_pca)
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained: {pca.explained_variance_ratio_.sum():.4f}")

# The key difference: PCA maximizes variance, AE minimizes reconstruction loss.
# With tied weights and ReLU decoder, the AE can learn a different projection
# that works better with the nonlinear decoder even if it captures less variance.

In [ ]:
# 3b: The AE's native decoder is W.T + bias + ReLU (tied weights).
# An MLP decoder with (64, 32) hidden units is much more expressive.
# Of course it does better! The AE is constrained to decode with W.T.
#
# The real question: given the SAME decoder capacity (MLP),
# which 2D projection gives better reconstruction?

x = datasets[5]
n_train = 1600
x_train, x_test = x[:n_train], x[n_train:]

# AE embedding
with torch.no_grad():
    z_ae_tr = model.encode(torch.tensor(x_train, dtype=torch.float32, device=device)).cpu().numpy()
    z_ae_te = model.encode(torch.tensor(x_test, dtype=torch.float32, device=device)).cpu().numpy()

# PCA embedding
pca = PCA(n_components=2)
z_pca_tr = pca.fit_transform(x_train)
z_pca_te = pca.transform(x_test)

# Compare with same MLP across multiple seeds
print("n=5: Same MLP decoder, different embeddings")
print(f"{'Embedding':<12} {'Test MSE (mean +/- std)'}")
print('-' * 40)
for name, z_tr, z_te in [('AE l=1', z_ae_tr, z_ae_te), ('PCA', z_pca_tr, z_pca_te)]:
    mses = []
    for seed in range(10):
        mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=seed,
                           early_stopping=True, validation_fraction=0.2)
        mlp.fit(z_tr, x_train)
        mse = np.mean((x_test - mlp.predict(z_te))**2)
        mses.append(mse)
    print(f"{name:<12} {np.mean(mses):.6f} +/- {np.std(mses):.6f}")

# Also compare: how correlated are the two embeddings?
print(f"\nCorrelation between AE and PCA embeddings (on test):")
for dim in range(2):
    corr = np.corrcoef(z_ae_te[:, dim], z_pca_te[:, dim])[0, 1]
    print(f"  Dim {dim}: r = {corr:.4f}")

# They should be similar since both are linear projections of the same data
# Check subspace alignment
from scipy.linalg import subspace_angles
angles = subspace_angles(W_ae.T, W_pca.T) * 180 / np.pi
print(f"\nSubspace angles between AE and PCA projection planes: {angles}")

## Surprise 4: Deep AE (l=3) worse than l=1 at n=5

l=3 native MSE=0.0045 vs l=1's MSE=0.0019. 

Is this:
- Optimization difficulty (deeper networks harder to train)?
- Overfitting (more parameters, same data)?
- Unnecessary capacity (linear encoder is already optimal at this compression ratio)?
- An artifact of importance weighting?

In [ ]:
# 4a: Is this consistent across seeds, or did we just get a bad seed for l=3?
print("Multiple seed runs for n=5, m=2:")
print(f"{'l':<5} {'Best loss':<14} {'Nonlinear gain':<16} {'Seeds tried'}")
print('-' * 50)
for l in [1, 2, 3, 4]:
    res = run_experiment_multi_seed(
        n=5, m=2, l=l, S=0.95, n_seeds=20, n_steps=10000,
        importance_decay=0.7, verbose=False
    )
    print(f"{l:<5} {res['final_loss']:<14.6f} {res['nonlinear_gain']:<16.4f} {res['seeds_tried']}")
    print(f"      Loss range: [{min(res['all_final_losses']):.6f}, {max(res['all_final_losses']):.6f}]")

In [ ]:
# 4b: Compare parameter counts and check for optimization issues
from core import Autoencoder

print("Parameter counts and architecture:")
for l in [1, 2, 3, 4]:
    model = Autoencoder(5, 2, l)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  l={l}: {n_params} parameters")

print("\nFor context: n=5, m=2")
print("  l=1 tied weights: W is (2,5) = 10 params + 5 bias = 15 params")
print("  The data has 5 features with 5% activation each")
print("  At S=0.95, avg ~0.25 features active per sample")
print("  n/m = 2.5 — modest compression, superposition barely needed")

In [ ]:
# 4c: Training curves — does l=3 converge or is it still improving?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for l in [1, 2, 3]:
    res = run_experiment_multi_seed(
        n=5, m=2, l=l, S=0.95, n_seeds=10, n_steps=20000,
        importance_decay=0.7, verbose=False
    )
    losses = res['losses']
    # Smooth
    window = 100
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
    axes[0].plot(smoothed, label=f'l={l}', alpha=0.8)
    axes[1].plot(smoothed[-5000:], label=f'l={l} (last 5k)', alpha=0.8)

axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Full training curves (n=5, m=2)')
axes[0].legend()
axes[0].set_yscale('log')

axes[1].set_xlabel('Step (offset)')
axes[1].set_ylabel('Loss')
axes[1].set_title('Last 5000 steps')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 4d: Does the l=3 advantage appear at higher compression ratios?
print("Depth comparison across compression ratios (best of 10 seeds):")
print(f"{'n':<5} {'m':<5} {'n/m':<6} {'l=1 loss':<14} {'l=3 loss':<14} {'l=3 better?'}")
print('-' * 60)

for n, m in [(5, 2), (8, 2), (16, 2), (32, 2), (64, 2)]:
    losses = {}
    for l in [1, 3]:
        res = run_experiment_multi_seed(
            n=n, m=m, l=l, S=0.95, n_seeds=10, n_steps=10000,
            importance_decay=0.7, verbose=False
        )
        losses[l] = res['final_loss']
    
    better = 'YES' if losses[3] < losses[1] else 'no'
    print(f"{n:<5} {m:<5} {n/m:<6.1f} {losses[1]:<14.6f} {losses[3]:<14.6f} {better}")

## Summary of findings

In [ ]:
# This cell will be filled in after running the above